# 🚀 CampaignX: Zero-Shot Claims Classifier Training & Export Notebook
Use this Google Colab Notebook to load, evaluate, or fine-tune **DistilBART / BART MNLI** on GPU for automated marketing compliance auditing.

Once exported, unzip the downloaded `claims-classifier.zip` into your local project directory at:
`backend/models/claims-classifier/`

In [ ]:
# 1. Install dependencies
!pip install -q transformers torch sentence-transformers datasets

In [ ]:
# 2. Check GPU Environment
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Running on CPU. Switch Colab runtime to GPU (Runtime -> Change runtime type -> T4 GPU) for fastest speed.")

In [ ]:
# 3. Model Configuration & Pipeline Setup
# Options:
# - "valhalla/distilbart-mnli-12-3" (Lightweight & Fast - ~300MB)
# - "facebook/bart-large-mnli" (High Accuracy - ~1.6GB)
MODEL_NAME = "valhalla/distilbart-mnli-12-3"
EXPORT_DIR = "claims-classifier"

print(f"Loading model: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("zero-shot-classification", model=model, tokenizer=tokenizer, device=device)
print("✅ Model successfully loaded on", "GPU (cuda:0)" if device == 0 else "CPU")

In [ ]:
# 4. Evaluate Zero-Shot Compliance Taxonomy Test
candidate_labels = [
    "unsubstantiated guarantee",
    "missing required disclaimer",
    "direct competitor comparison",
    "medical or health claim",
    "financial or earnings claim",
    "neutral marketing statement"
]

test_copy = "We guarantee 100% risk-free 10x ROI overnight without any effort."
res = classifier(test_copy, candidate_labels)

print(f"Test Copy: \"{test_copy}\"")
print(f"Top Flag: {res['labels'][0]} (Confidence: {res['scores'][0]*100:.2f}%)")
print("\nFull Label Probabilities:")
for label, score in zip(res['labels'], res['scores']):
    print(f" - {label}: {score*100:.2f}%")

In [ ]:
# 5. Save & Package Model for CampaignX Local Backend
os.makedirs(EXPORT_DIR, exist_ok=True)
tokenizer.save_pretrained(EXPORT_DIR)
model.save_pretrained(EXPORT_DIR)
print(f"Saved exported weights to folder: '{EXPORT_DIR}'")

# Zip directory for easy download
!zip -r claims-classifier.zip claims-classifier/
print("\n📦 'claims-classifier.zip' created successfully!")
print("Download 'claims-classifier.zip' from the Colab left sidebar and extract to backend/models/claims-classifier/")